# YOLO PA H&E Two-Class Boundary Fine-Tuning

Run this notebook on the GPU cluster after cloning two separate repos:

- training repo: `nttssv/cell_count`, branch `codex/yolo-cluster-training-package-github`
- data repo: `nttssv/training_pa_he_annotation`

The notebook reads the data repo from `DATA_REPO_ROOT`, builds a runtime YOLO dataset under this training repo's `outputs/` folder, and trains a two-class segmentation model:

- `nucleus`
- `cell_boundary` = `clear_cell_boundary` + `GT uncertain cell boundary`

`compact_cell_boundary` and `stroma` are intentionally dropped for this model.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, clear_output, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'training/run_yolo_segment_train.py').exists() and (candidate / 'training/build_pa_he_2class_dataset.py').exists():
            return candidate
    raise FileNotFoundError('Could not find cell_count training repo root from current path')


def has_nvidia_gpu() -> bool:
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False


def choose_training_model(output_root: Path, reference_model_dir: Path, repo_root: Path) -> Path | str:
    candidates = []
    if output_root.exists():
        candidates.extend(sorted(output_root.glob('*/weights/best.pt'), key=lambda p: p.stat().st_mtime, reverse=True))
    candidates.extend([
        reference_model_dir / 'yolo_pa_he_2class_boundary_plus_uncertain_best.pt',
        reference_model_dir / 'yolo_2class_nucleus_clear_boundary_precision_best.pt',
        reference_model_dir / 'yolo_sam31_p2_24tiles_best.pt',
        reference_model_dir / 'cellseg1_cgh_p2_yolo_best.pt',
        repo_root / 'yolov8s-seg.pt',
    ])
    for candidate in candidates:
        if isinstance(candidate, Path) and candidate.exists():
            return candidate
    return 'yolov8s-seg.pt'


REPO_ROOT = find_repo_root(Path.cwd())
DATA_REPO_URL = 'https://github.com/nttssv/training_pa_he_annotation.git'
DATA_REPO_ROOT = Path(os.environ.get('PA_HE_DATA_REPO', Path.home() / 'Desktop/training_pa_he_annotation')).expanduser()
SOURCE_YOLO_DATASET = DATA_REPO_ROOT / 'yolo_seg_dataset'
SOURCE_AUX_MASKS = DATA_REPO_ROOT / 'auxiliary_masks'
OUTPUT_ROOT = REPO_ROOT / 'outputs/yolo_cluster_live'
REFERENCE_MODEL_DIR = REPO_ROOT / 'training_data/reference_models'
EXPERIMENT_NAME = 'pa_he_2class_boundary_plus_uncertain'
YOLO_DATASET = OUTPUT_ROOT / 'datasets' / EXPERIMENT_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: 'nucleus', 1: 'cell_boundary'}

# Mild train-only oversampling for dense/hard examples. Edit or empty this dict if needed.
OVERSAMPLE_MULTIPLIERS = {
    'p2_tile_12': 2,
    'p2_tile_14': 2,
    'p2_tile_16': 2,
    'yolo_tile_21': 2,
}

EPOCHS = 180
IMGSZ = 512
BATCH = 8
WORKERS = 0
PATIENCE = 45
DEVICE = '0' if has_nvidia_gpu() else 'cpu'
MODEL = choose_training_model(OUTPUT_ROOT, REFERENCE_MODEL_DIR, REPO_ROOT)
FINE_TUNING = isinstance(MODEL, Path) and MODEL.exists() and MODEL.name != 'yolov8s-seg.pt'
LR0 = 0.00025 if FINE_TUNING else 0.001
RUN_NAME = 'yolo_pa_he_2class_uncertain_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUT_ROOT / RUN_NAME
LIVE_LOG = OUTPUT_ROOT / f'{RUN_NAME}_live_training.log'
RUNTIME_DATA_YAML = YOLO_DATASET / 'data.yaml'
PRED_CONF = 0.45
PRED_IOU = 0.40
PRED_MAX_DET = 100

print('REPO_ROOT:', REPO_ROOT)
print('DATA_REPO_ROOT:', DATA_REPO_ROOT)
print('DATA_REPO_URL:', DATA_REPO_URL)
print('SOURCE_YOLO_DATASET:', SOURCE_YOLO_DATASET)
print('SOURCE_AUX_MASKS:', SOURCE_AUX_MASKS)
print('GENERATED_YOLO_DATASET:', YOLO_DATASET)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('MODEL:', MODEL)
print('FINE_TUNING:', FINE_TUNING)
print('LR0:', LR0)
print('DEVICE:', DEVICE)
print('WORKERS:', WORKERS)
print('RUN_NAME:', RUN_NAME)
print('PRED_CONF:', PRED_CONF)
print('PRED_IOU:', PRED_IOU)


In [ ]:
try:
    import ultralytics
    print('ultralytics:', ultralytics.__version__)
except Exception as exc:
    raise RuntimeError('Ultralytics is not installed. Install once with: python -m pip install ultralytics') from exc

if not DATA_REPO_ROOT.exists():
    raise FileNotFoundError(
        f'Data repo not found: {DATA_REPO_ROOT}\n'
        f'Clone it first on the cluster:\n'
        f'  cd ~/Desktop && git clone {DATA_REPO_URL}\n'
        f'Or set PA_HE_DATA_REPO to a different path before running this notebook.'
    )
assert SOURCE_YOLO_DATASET.exists(), SOURCE_YOLO_DATASET
assert SOURCE_AUX_MASKS.exists(), SOURCE_AUX_MASKS

DATASET_BUILDER = REPO_ROOT / 'training/build_pa_he_2class_dataset.py'
assert DATASET_BUILDER.exists(), DATASET_BUILDER

cmd = [
    sys.executable, str(DATASET_BUILDER),
    '--data-repo', str(DATA_REPO_ROOT),
    '--output', str(YOLO_DATASET),
    '--include-uncertain-as-boundary',
]
for tile_id, multiplier in OVERSAMPLE_MULTIPLIERS.items():
    cmd.extend(['--oversample-tile', f'{tile_id}={multiplier}'])

print('Dataset build command:')
print(' '.join(shlex.quote(str(x)) for x in cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Dataset builder failed with return code {result.returncode}')

assert RUNTIME_DATA_YAML.exists(), RUNTIME_DATA_YAML
summary_csv = YOLO_DATASET / 'label_counts_2class_boundary_plus_uncertain.csv'
summary_json = YOLO_DATASET / 'conversion_summary_2class_boundary_plus_uncertain.json'
assert summary_csv.exists(), summary_csv
assert summary_json.exists(), summary_json

summary = pd.read_csv(summary_csv)
image_files = sorted((YOLO_DATASET / 'images').glob('*/*.png'))
label_files = sorted((YOLO_DATASET / 'labels').glob('*/*.txt'))
assert len(image_files) == len(label_files), (len(image_files), len(label_files))

print(RUNTIME_DATA_YAML)
print(RUNTIME_DATA_YAML.read_text())
print('Images:', len(image_files), 'Labels:', len(label_files))
display(summary.groupby('split')[['nucleus', 'clear_cell_boundary', 'uncertain_cell_boundary', 'cell_boundary', 'total']].sum())
display(summary)


In [ ]:
def nvidia_snapshot() -> str:
    try:
        result = subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
            '--format=csv,noheader,nounits',
        ], capture_output=True, text=True, timeout=5)
        return result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    except Exception as exc:
        return f'nvidia-smi unavailable: {exc}'

def tail_text(path: Path, lines: int = 18) -> str:
    if not path.exists():
        return '(log not created yet)'
    text = path.read_text(errors='replace')
    return '\\n'.join(text.splitlines()[-lines:])

def load_results(run_dir: Path):
    path = run_dir / 'results.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

def plot_live_results(df: pd.DataFrame, out_path: Path):
    if df is None or df.empty:
        return
    x = df['epoch'] if 'epoch' in df.columns else range(len(df))
    loss_cols = [c for c in df.columns if 'loss' in c.lower()]
    metric_cols = [c for c in df.columns if c.startswith('metrics/') or 'mAP' in c or 'precision' in c or 'recall' in c]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for col in loss_cols:
        axes[0].plot(x, df[col], label=col.replace('train/', '').replace('val/', 'val '))
    axes[0].set_title('YOLO losses')
    axes[0].set_xlabel('epoch')
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    for col in metric_cols:
        axes[1].plot(x, df[col], label=col.replace('metrics/', ''))
    axes[1].set_title('Validation metrics')
    axes[1].set_xlabel('epoch')
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()

def live_dashboard(run_dir: Path, log_path: Path):
    clear_output(wait=True)
    print('Run:', run_dir)
    print('Time:', datetime.now().isoformat(timespec='seconds'))
    print('GPU:', nvidia_snapshot())
    df = load_results(run_dir)
    if df is not None and not df.empty:
        display(df.tail(5))
        plot_live_results(df, run_dir / 'live_metrics.png')
    else:
        print('Waiting for results.csv...')
    print('--- log tail ---')
    print(tail_text(log_path))

In [ ]:
YOLO_RUNNER = REPO_ROOT / 'training/run_yolo_segment_train.py'
assert YOLO_RUNNER.exists(), YOLO_RUNNER

cmd = [
    sys.executable, str(YOLO_RUNNER),
    f'model={MODEL}',
    f'data={RUNTIME_DATA_YAML}',
    f'epochs={EPOCHS}',
    f'imgsz={IMGSZ}',
    f'batch={BATCH}',
    f'device={DEVICE}',
    f'workers={WORKERS}',
    f'patience={PATIENCE}',
    'optimizer=AdamW',
    f'lr0={LR0}',
    'mosaic=0.0',
    'close_mosaic=0',
    'copy_paste=0.0',
    'degrees=2',
    'translate=0.02',
    'scale=0.10',
    'fliplr=0.5',
    'mixup=0.0',
    'hsv_h=0.01',
    'hsv_s=0.25',
    'hsv_v=0.20',
    'erasing=0.0',
    'overlap_mask=False',
    'mask_ratio=2',
    'cls=1.0',
    f'project={OUTPUT_ROOT}',
    f'name={RUN_NAME}',
]

print('Command:')
print(' '.join(shlex.quote(str(x)) for x in cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('WANDB_MODE', 'offline')

with LIVE_LOG.open('w', encoding='utf-8') as log_handle:
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=log_handle, stderr=subprocess.STDOUT, text=True, env=env)
    while proc.poll() is None:
        live_dashboard(RUN_DIR, LIVE_LOG)
        time.sleep(15)
    return_code = proc.wait()

live_dashboard(RUN_DIR, LIVE_LOG)
if return_code != 0:
    raise RuntimeError(f'YOLO training failed with return code {return_code}. See {LIVE_LOG}')
print('Training complete')

In [ ]:
best_pt = RUN_DIR / 'weights/best.pt'
last_pt = RUN_DIR / 'weights/last.pt'
deployed_pt = REFERENCE_MODEL_DIR / 'yolo_pa_he_2class_boundary_plus_uncertain_best.pt'
if best_pt.exists():
    shutil.copy2(best_pt, deployed_pt)

summary = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'repo_root': str(REPO_ROOT),
    'data_repo_root': str(DATA_REPO_ROOT),
    'source_dataset': str(SOURCE_YOLO_DATASET),
    'source_auxiliary_masks': str(SOURCE_AUX_MASKS),
    'training_dataset': str(YOLO_DATASET),
    'runtime_data_yaml': str(RUNTIME_DATA_YAML),
    'run_dir': str(RUN_DIR),
    'live_log': str(LIVE_LOG),
    'best_pt': str(best_pt),
    'last_pt': str(last_pt),
    'deployed_pt': str(deployed_pt) if best_pt.exists() else '',
    'model_start': str(MODEL),
    'fine_tuning': FINE_TUNING,
    'classes': CLASS_NAMES,
    'boundary_source': 'clear_cell_boundary + GT uncertain cell boundary',
    'dropped_source_classes': {'2': 'compact_cell_boundary', '3': 'stroma'},
    'oversample_multipliers': OVERSAMPLE_MULTIPLIERS,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'device': DEVICE,
    'lr0': LR0,
    'prediction_conf': PRED_CONF,
    'prediction_iou': PRED_IOU,
    'prediction_max_det': PRED_MAX_DET,
    'augmentation': {
        'mosaic': 0.0,
        'close_mosaic': 0,
        'copy_paste': 0.0,
        'degrees': 2,
        'translate': 0.02,
        'scale': 0.10,
        'fliplr': 0.5,
        'mixup': 0.0,
        'hsv_h': 0.01,
        'hsv_s': 0.25,
        'hsv_v': 0.20,
        'erasing': 0.0,
        'overlap_mask': False,
        'mask_ratio': 2,
        'cls': 1.0,
    },
}
(RUN_DIR / 'yolo_live_training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(Markdown('## Final files'))
for key, value in summary.items():
    print(f'{key}: {value}')
print('Available plots:', [p.name for p in RUN_DIR.glob('*.png')])


In [ ]:
# Qualitative check after training: original vs merged ground truth vs YOLO prediction.
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from ultralytics import YOLO

assert best_pt.exists(), best_pt
VAL_IMG_DIR = YOLO_DATASET / 'images/val'
VAL_LBL_DIR = YOLO_DATASET / 'labels/val'
COMPARE_DIR = RUN_DIR / f'comparison_original_gt_pred_conf{PRED_CONF:.2f}_iou{PRED_IOU:.2f}'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_COLORS = {
    0: (30, 120, 255),   # nucleus: blue in RGB
    1: (0, 180, 120),    # cell boundary: green in RGB
}
CLASS_NAMES = dict(CLASS_NAMES)


def read_rgb(path: Path) -> np.ndarray:
    img = cv2.imread(str(path))
    assert img is not None, path
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def draw_yolo_seg_labels(img_rgb: np.ndarray, label_path: Path, alpha: float = 0.35) -> np.ndarray:
    out = img_rgb.copy()
    overlay = img_rgb.copy()
    h, w = img_rgb.shape[:2]

    if not label_path.exists():
        return out

    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        coords = list(map(float, parts[1:]))
        pts = []
        for x, y in zip(coords[0::2], coords[1::2]):
            pts.append([int(x * w), int(y * h)])
        if len(pts) < 3:
            continue
        pts = np.array(pts, dtype=np.int32)
        color = CLASS_COLORS.get(cls, (255, 255, 0))
        cv2.fillPoly(overlay, [pts], color)
        cv2.polylines(out, [pts], isClosed=True, color=color, thickness=2)

    return cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)


def draw_prediction(img_rgb: np.ndarray, result, alpha: float = 0.35) -> np.ndarray:
    pred = img_rgb.copy()
    if result.masks is None:
        return pred

    overlay = img_rgb.copy()
    masks = result.masks.data.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy() if result.boxes.conf is not None else np.ones(len(classes))

    for mask, cls, conf in zip(masks, classes, confs):
        mask = cv2.resize(mask, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
        mask_bin = mask > 0.5
        if mask_bin.sum() < 12:
            continue
        color = np.array(CLASS_COLORS.get(cls, (255, 255, 0)), dtype=np.uint8)
        overlay[mask_bin] = color
        contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(pred, contours, -1, tuple(map(int, color)), 2)
        if contours:
            x, y, _, _ = cv2.boundingRect(max(contours, key=cv2.contourArea))
            cv2.putText(pred, f'{CLASS_NAMES.get(cls, cls)} {conf:.2f}', (x, max(12, y - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.36, tuple(map(int, color)), 1, cv2.LINE_AA)

    return cv2.addWeighted(overlay, alpha, pred, 1 - alpha, 0)


model = YOLO(str(best_pt))
pred_results = model.predict(
    source=str(VAL_IMG_DIR),
    imgsz=IMGSZ,
    conf=PRED_CONF,
    iou=PRED_IOU,
    max_det=PRED_MAX_DET,
    save=False,
    verbose=False,
)

saved = []
for result in pred_results:
    img_path = Path(result.path)
    tile_id = img_path.stem
    original = read_rgb(img_path)
    gt = draw_yolo_seg_labels(original, VAL_LBL_DIR / f'{tile_id}.txt')
    pred = draw_prediction(original, result)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = [original, gt, pred]
    titles = [
        f'{tile_id}: original',
        'merged ground truth labels',
        f'YOLO prediction conf={PRED_CONF:.2f}, iou={PRED_IOU:.2f}',
    ]
    for ax, im, title in zip(axes, panels, titles):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis('off')

    handles = [
        plt.Line2D([0], [0], color=np.array(CLASS_COLORS[i]) / 255, lw=4, label=CLASS_NAMES[i])
        for i in sorted(CLASS_NAMES)
    ]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles))
    fig.tight_layout(rect=[0, 0.06, 1, 1])

    out_path = COMPARE_DIR / f'{tile_id}_original_gt_pred.png'
    fig.savefig(out_path, dpi=180)
    plt.close(fig)
    saved.append(out_path)
    display(Image(filename=str(out_path)))

print('Saved comparisons to:', COMPARE_DIR)
print('Files:')
for path in saved:
    print(path)
